In [2]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler, PolynomialFeatures
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error


In [3]:
df = pd.read_csv("insurance.csv")

print("Initial shape:", df.shape)

Initial shape: (1338, 7)


In [4]:
df_drop = df.dropna()

In [5]:
Q1 = df.quantile(0.25, numeric_only=True)
Q3 = df.quantile(0.75, numeric_only=True)
IQR = Q3 - Q1

df = df[~((df.select_dtypes(include=[np.number]) < (Q1 - 1.5 * IQR)) | 
          (df.select_dtypes(include=[np.number]) > (Q3 + 1.5 * IQR))).any(axis=1)]


In [6]:
df = pd.get_dummies(df, drop_first=True)

In [7]:
X = df.drop("charges", axis=1)
y = df["charges"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)


In [8]:
scaler = MinMaxScaler()

X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [9]:
lr = LinearRegression()
lr.fit(X_train, y_train)

train_pred = lr.predict(X_train)
test_pred = lr.predict(X_test)

print("\n--- Linear Regression ---")
print("Train MSE:", mean_squared_error(y_train, train_pred))
print("Test MSE:", mean_squared_error(y_test, test_pred))



--- Linear Regression ---
Train MSE: 21395298.055043772
Test MSE: 19219976.00767453


In [10]:
print("\n--- Polynomial Regression ---")

best_degree = 1
best_error = float('inf')

for d in range(2, 6):
    poly = PolynomialFeatures(degree=d)
    
    X_train_poly = poly.fit_transform(X_train)
    X_test_poly = poly.transform(X_test)
    
    model = LinearRegression()
    model.fit(X_train_poly, y_train)
    
    train_pred = model.predict(X_train_poly)
    test_pred = model.predict(X_test_poly)
    
    train_mse = mean_squared_error(y_train, train_pred)
    test_mse = mean_squared_error(y_test, test_pred)
    
    print(f"Degree {d}: Train MSE={train_mse}, Test MSE={test_mse}")
    
    if test_mse < best_error:
        best_error = test_mse
        best_degree = d

print("Best Polynomial Degree:", best_degree)



--- Polynomial Regression ---
Degree 2: Train MSE=18793848.471369695, Test MSE=17936496.932131145
Degree 3: Train MSE=16946620.286353227, Test MSE=20050475.11424724
Degree 4: Train MSE=14681855.549400216, Test MSE=57204871.08222898
Degree 5: Train MSE=12150311.356029214, Test MSE=15633004602.99117
Best Polynomial Degree: 2


In [11]:
print("\n--- Random Forest ---")

best_model = None
best_error = float('inf')

for depth in [3, 5, 10]:
    for features in ['sqrt', 'log2', None]:
        for criterion in ['squared_error', 'absolute_error']:
            
            rf = RandomForestRegressor(
                n_estimators=100,
                max_depth=depth,
                max_features=features,
                criterion=criterion,
                random_state=42
            )
            
            rf.fit(X_train, y_train)
            
            test_pred = rf.predict(X_test)
            test_mse = mean_squared_error(y_test, test_pred)
            
            if test_mse < best_error:
                best_error = test_mse
                best_model = rf



--- Random Forest ---


In [12]:
train_pred = best_model.predict(X_train)
test_pred = best_model.predict(X_test)

print("Best Random Forest Train MSE:", mean_squared_error(y_train, train_pred))
print("Best Random Forest Test MSE:", mean_squared_error(y_test, test_pred))

Best Random Forest Train MSE: 15699628.914456856
Best Random Forest Test MSE: 17009975.32835077
